# Автоматическое дифференцирование

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Сравнивать autograd с формулой градиента и управлять накоплением производных.

## Данные примера

Два объекта имеют по два числовых признака. Правильные ответы — 5 и 4. Все вычисления выполняются на CPU, центральном процессоре компьютера.

In [ ]:
import torch
X = torch.tensor([[2., 1.], [1., 3.]], dtype=torch.float64)
y = torch.tensor([5., 4.], dtype=torch.float64)
w = torch.tensor([1., 1.], dtype=torch.float64, requires_grad=True)
prediction = X @ w


## Граф вычислений и autograd

Autograd означает автоматическое дифференцирование: библиотека применяет известные правила производных к выполненным операциям, а не угадывает их по соседним точкам. `requires_grad=True` включает запись операций над параметром для автоматического дифференцирования. Граф связывает входы и промежуточные результаты. Для скалярной ошибки `backward()` применяет правило цепочки и записывает производные в `.grad`.

Скаляр — одно число; ошибка должна быть скалярной для вызова `backward()` без дополнительных аргументов. Метод `mean()` вычисляет среднее, `X.T` транспонирует таблицу, `torch.allclose` сравнивает тензоры с допуском на численные погрешности.

`detach()` даёт тензор без отслеживания градиента, но может разделять память с исходным. Для независимой копии используйте `.detach().clone()`. `.item()` превращает тензор из одного элемента в обычное число — удобно для журнала, но через такое число градиент уже не проходит.

## Разобранный пример: градиент ошибки

Сначала вычислим скалярную ошибку, затем её производные. Сравним автоматический результат с формулой из математического раздела.

In [ ]:
loss = ((prediction - y) ** 2).mean()
loss.backward()
expected_gradient = 2 / len(y) * X.T @ (X @ w.detach() - y)
assert torch.allclose(w.grad, expected_gradient)
assert torch.allclose(w.grad, torch.tensor([-4., -2.], dtype=torch.float64))
print(loss.item())  # 2.0


У первого объекта знаковая ошибка −2, у второго 0; средний квадрат равен 2. Обозначим знаковую ошибку первого объекта через `e₁`, второго — через `e₂`. Средняя квадратичная ошибка `L = (e₁² + e₂²) / 2`. Производная по первому весу равна `2 / 2 * (e₁ * 2 + e₂ * 1) = -4`, по второму — `2 / 2 * (e₁ * 1 + e₂ * 3) = -2`. Числа 2, 1 и 1, 3 — соответствующие признаки объектов. Матрица `X.T` в коде собирает именно эти суммы. `backward` вычисляет их, но сам не меняет веса.

## Накопление и отключение градиентов

Повторные `backward` складывают градиенты, а не заменяют их. В обычном шаге обучения сначала обнуляют старые значения, заново вычисляют ошибку и вызывают `backward`. Старый граф после обратного прохода обычно освобождается, поэтому просто повторить `loss.backward()` нельзя.

Конструкция `with torch.no_grad():` задаёт блок с временно отключённой записью графа; после блока прежняя настройка восстанавливается. `torch.no_grad()` временно отключает построение графа. Так можно обновлять параметры или получать проверочные ответы без лишней памяти. Обновление здесь — вычисление нового набора весов `w - 0.01 * w.grad`, где 0.01 задаёт размер шага. Оно записывается в новое имя `updated`, исходный `w` не меняется. В полноценном обучении обновления поручают оптимизатору — объекту, выполняющему шаги изменения параметров; его разберём вместе с нейросетями.

In [ ]:
w.grad = None
fresh_loss = ((X @ w - y) ** 2).mean()
fresh_loss.backward()
assert torch.allclose(w.grad, expected_gradient)
with torch.no_grad():
    updated = w - 0.01 * w.grad
assert not updated.requires_grad


## Задачи

### 1. Базовая

Обновляет ли backward сами веса?

<details>
<summary>Подсказка</summary>

Он вычисляет производные.

</details>



### 2. Базовая

Что означает requires_grad=True?

<details>
<summary>Подсказка</summary>

По этому тензору понадобятся производные.

</details>



### 3. Прикладная

Проверьте производную (z−1)² при z=3.

<details>
<summary>Подсказка</summary>

Примените правило производной квадрата к внутреннему выражению z−1.

</details>



### 4. Прикладная

Обнулите градиент z и заново вычислите ту же производную.

<details>
<summary>Подсказка</summary>

Сначала grad = None, затем новое выражение.

</details>



### 5. Олимпиадная
У `z` уже записан градиент 4 для `(z−1)²` при `z = 3`. Не обнуляя `z.grad`, заново постройте выражение `(z - 1) ** 2` и вызовите у него `backward()`. Каким станет градиент? Чем это отличается от повторного обхода одного уже освобождённого графа?

<details>
<summary>Подсказка</summary>

Новый граф вычисляет ещё один вклад; подумайте, куда PyTorch его запишет.

</details>



### 6. Олимпиадная

Почему нельзя вызвать backward у результата loss.item()?

<details>
<summary>Подсказка</summary>

Это уже обычное число.

</details>



## Итоги

Вы умеете получать градиент скалярной ошибки и проверять его ручным вычислением.

## Самопроверка

Что меняет `backward`: веса или их градиенты? Почему перед обычным новым шагом градиенты обнуляют? Чем `detach` отличается от независимой копии?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Нет, он записывает градиенты.

### Задача 2

Включает отслеживание операций для вычисления градиента.

### Задача 3

In [ ]:
z = torch.tensor(3., dtype=torch.float64, requires_grad=True)
((z - 1) ** 2).backward()
assert z.grad.item() == 4


### Задача 4

In [ ]:
z.grad = None
((z - 1) ** 2).backward()
assert z.grad.item() == 4


### Задача 5

Градиенты накапливаются, поэтому к 4 добавится ещё 4: получится 8. Здесь выражение строится заново, значит создаётся новый граф. Вызов `backward()` второй раз у той же прежней потери без сохранения графа обычно вызовет ошибку: нужные промежуточные данные уже освобождены.

In [ ]:
z = torch.tensor(3., dtype=torch.float64, requires_grad=True)
first_loss = (z - 1) ** 2
first_loss.backward()
assert z.grad.item() == 4
second_loss = (z - 1) ** 2
second_loss.backward()
assert z.grad.item() == 8


Для независимых шагов обучения градиенты очищают перед новым вычислением.

### Задача 6

Оно не хранит граф вычислений. Для обратного прохода нужен тензор ошибки.